# Building Autonomous AI Agents — Day 2
## Final Lab — A2A  - Remote Agent Collaboration with A2A


The Coordinator does not attach each remote agent as a separate tool.
Instead, it:

1. discovers remote agents from their **Agent Cards**;
2. reasons over the discovered agents and their capabilities;
3. selects the agent to delegate to;
4. uses **one generic delegation tool** to carry out the A2A call.

```text
                   Agent Cards
                       │
                       ▼
                Coordinator Agent
                       │
               selects agent + task
                       │
                       ▼
             ask_remote_agent(...)
                       │
                      A2A
                       │
                       ▼
             Selected Remote Agent
```

**Key idea:** the remote agent is still an autonomous agent.
The tool is only the delegation/transport mechanism.

# Part A — Setup

## A1. Install packages

In [1]:
%pip install -q "langchain>=1.0,<2.0" langchain-openai "a2a-sdk[http-server]==0.3.26" "httpx>=0.28,<1.0"

!wget -q \
  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
  -O /usr/local/bin/cloudflared

!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.0/151.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.4/146.4 kB 2.3 MB/s eta 0:00:00
cloudflared version 2026.10.3 (built 2026-10-21-13:11 UTC)


## A2. Mount Google Drive and reuse Day 1

In [2]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import sys

COURSE_ROOT = Path("/content/drive/MyDrive/BAAI_GoogleDrive")
DAY1_SRC = COURSE_ROOT / "Day1-Workshop-Starter" / "src"
AGENTMART_DB_PATH = COURSE_ROOT / "Day1-Workshop-Starter" / "data" / "agentmart.db"

TOOLS_FILE = DAY1_SRC / "agentmart" / "tools.py"

if not TOOLS_FILE.exists():
    raise FileNotFoundError(
        f"Cannot find Day 1 tools at: {TOOLS_FILE}\n"
        "Check COURSE_ROOT and the Google Drive folder structure."
    )

if str(DAY1_SRC) not in sys.path:
    sys.path.insert(0, str(DAY1_SRC))

os.environ["AGENTMART_DB_PATH"] = str(AGENTMART_DB_PATH)

print("Course root :", COURSE_ROOT)
print("Day 1 source:", DAY1_SRC)
print("SQLite DB   :", AGENTMART_DB_PATH)

Mounted at /content/drive
Course root : /content/drive/MyDrive/BAAI_GoogleDrive
Day 1 source: /content/drive/MyDrive/BAAI_GoogleDrive/Day1-Workshop-Starter/src
SQLite DB   : /content/drive/MyDrive/BAAI_GoogleDrive/Day1-Workshop-Starter/data/agentmart.db


## A3. Configure the OpenAI model

In [3]:
import getpass

api_key = None

try:
    from google.colab import userdata
    api_key = userdata.get("OPENROUTER_API_KEY")
except Exception:
    pass

if not api_key:
    api_key = getpass.getpass("Enter OPENROUTER_API_KEY: ")

os.environ["OPENROUTER_API_KEY"] = api_key
os.environ["OPENAI_MODEL"]="openai/gpt-4o-mini"

model = os.getenv("OPENAI_MODEL")

print("Model:", os.getenv("OPENAI_MODEL"))
print("OpenRouter API key configured:", bool(os.environ.get("OPENROUTER_API_KEY")))

Model: openai/gpt-4o-mini
OpenRouter API key configured: True


## A4. Import Day 1 database and tools

In [4]:
from agentmart.database import initialize_database
from agentmart.tools import (
    search_products,
    get_product_details,
    check_inventory,
    calculate_discount,
    estimate_delivery,
)

DB_PATH = initialize_database(reset=False)

print("AgentMart database:", DB_PATH)
print("Day 1 tools imported successfully.")

AgentMart database: /content/drive/MyDrive/BAAI_GoogleDrive/Day1-Workshop-Starter/data/agentmart.db
Day 1 tools imported successfully.


## A5. Common helpers

In [5]:
import re

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI


llm = ChatOpenAI(
    model= model,
    api_key= api_key,
    base_url="https://openrouter.ai/api/v1",
    temperature=0.4,
    max_retries=2
)


DEFAULT_QUERY = (
    "I need a pair of noise-cancelling headphones below S$400. "
    "I travel frequently, so battery life matters. "
    "Recommend one and tell me whether it is available."
)


CHALLENGE_QUERY = (
    "I need noise-cancelling headphones below S$400. "
    "Recommend the eligible product with the highest current discount "
    "and check whether it is in stock. "
    "If it is out of stock, do not stop—exclude that product and "
    "recommend the next-highest-discount eligible product, then check "
    "stock again. Continue until you find an in-stock product. Do not check the same product twice, and stop after at most 3 candidates."
)


print("Common helpers ready.")

Common helpers ready.


## A.6 Build the same two specialist agents

These prompts and agent definitions are intentionally the same as Checkpoints 1 and 2.

**Only the deployment and orchestration boundary changes.**

In [6]:
PRODUCT_ADVISOR_PROMPT = """You are AgentMart's Product Advisor Agent.

Your responsibility is product discovery, comparison, recommendation, and pricing.
Use tools for every AgentMart fact. Never invent specifications or prices.

Rules:
- Search before recommending.
- Treat explicit category, budget, and feature constraints as mandatory.
- Compare the attributes the customer says matter.
- Do NOT check live inventory.
- Do NOT estimate delivery.
- Keep the response concise.
"""


FULFILMENT_PROMPT = """You are AgentMart's Fulfilment Agent.

Your responsibility is live availability and Singapore delivery only.
Use tools for every AgentMart fact. Never invent inventory or delivery estimates.

Rules:
- If delivery is requested without a postal code, ask for one.
- Do not recommend products or compare product specifications.
- Keep the response concise and evidence based.
"""

from pydantic import BaseModel
class ProductAdvisorResponse(BaseModel):
    answer: str
    product_id: str | None = None

product_advisor = create_agent(
    model=llm,
    tools=[search_products, get_product_details, calculate_discount],
    name="product_advisor",
    system_prompt=PRODUCT_ADVISOR_PROMPT,
    response_format=ProductAdvisorResponse
)


class FulfilmentResponse(BaseModel):
    answer: str
    in_stock: bool | None = None

fulfilment_agent = create_agent(
    model=llm,
    tools=[check_inventory, estimate_delivery],
    name="fulfilment_agent",
    system_prompt=FULFILMENT_PROMPT,
    response_format=FulfilmentResponse
)

print("Specialist agents ready.")

Specialist agents ready.


# Part B — Expose Both Specialists as Independent A2A Agents

## 3.1 Import A2A and server libraries

In [7]:
import json
import socket
import subprocess
import threading
import time
from typing import Any, Dict
from uuid import uuid4

import httpx
import uvicorn

from a2a.client import A2ACardResolver, A2AClient
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.apps import A2AStarletteApplication
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentSkill,
    MessageSendParams,
    SendMessageRequest,
)
from a2a.utils import new_agent_text_message


PRODUCT_PORT = 9998
FULFILMENT_PORT = 9999

## 3.2 Wrap each local specialist in an A2A `AgentExecutor`

In [25]:
async def run_product_advisor_async(task):
    result = await product_advisor.ainvoke(
        {"messages": [{"role": "user", "content": task}]}
    )
    response = result["structured_response"]
    return response.model_dump_json()


async def run_fulfilment_async(task):
    result = await fulfilment_agent.ainvoke(
        {"messages": [{"role": "user", "content": task}]}
    )
    response = result["structured_response"]
    return response.model_dump_json()


class ProductAdvisorAgentExecutor(AgentExecutor):
    async def execute(
        self,
        context: RequestContext,
        event_queue: EventQueue,
    ) -> None:
        query = context.get_user_input()

        print("\n=== Product Advisor A2A ===")
        print("query:", query)

        reply_text = await run_product_advisor_async(query)

        print("response:", reply_text)

        await event_queue.enqueue_event(
            new_agent_text_message(reply_text)
        )

    async def cancel(
        self,
        context: RequestContext,
        event_queue: EventQueue,
    ) -> None:
        raise Exception("cancel not supported")

In [24]:
#To Do: complete the class FulfilmentAgentExecutor(AgentExecutor)
class FulfilmentAgentExecutor(AgentExecutor):
    async def execute(
    self,
    context: RequestContext,
    event_queue: EventQueue,
    ) -> None:
      query = context.get_user_input()
      print("\n=== Fulfilment A2A ===")
      print("query:", query)
      reply_text = await run_fulfilment_async(query)
      print("response:", reply_text)
      await event_queue.enqueue_event(
      new_agent_text_message(reply_text)
      )
    async def cancel(
      self,
      context: RequestContext,
      event_queue: EventQueue,
    ) -> None:
      raise Exception("cancel not supported")


## 3.3 Create Cloudflare Quick Tunnels

In [22]:
A2A_SERVERS = globals().get("A2A_SERVERS", {})
A2A_THREADS = globals().get("A2A_THREADS", {})
TUNNEL_PROCESSES = globals().get("TUNNEL_PROCESSES", {})


def start_quick_tunnel(
    name: str,
    port: int,
    timeout_seconds: int = 30,
):
    log_path = Path(f"/tmp/cloudflared-{name}.log")

    command = [
        "cloudflared",
        "tunnel",
        "--url",
        f"http://127.0.0.1:{port}",
        "--no-autoupdate",
    ]

    with open(log_path, "w") as log_file:
        process = subprocess.Popen(
            command,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            text=True,
        )

    url_pattern = re.compile(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com"
    )

    deadline = time.time() + timeout_seconds

    while time.time() < deadline:
        if process.poll() is not None:
            log_text = (
                log_path.read_text(errors="ignore")
                if log_path.exists()
                else ""
            )
            raise RuntimeError(
                "cloudflared stopped unexpectedly.\n" + log_text
            )

        if log_path.exists():
            log_text = log_path.read_text(errors="ignore")
            match = url_pattern.search(log_text)

            if match:
                public_url = match.group(0).rstrip("/")
                TUNNEL_PROCESSES[name] = process
                return process, public_url

        time.sleep(0.5)

    process.terminate()
    raise TimeoutError(
        "Timed out waiting for Cloudflare Quick Tunnel URL."
    )

## 3.4 Build Agent Cards

In [21]:
def build_product_agent_card(public_url: str):
    skill = AgentSkill(
        id="product_advisor",
        name="Product Advisor",
        description=(
            "Searches AgentMart products, compares product details, "
            "recommends products, and works with pricing and discounts."
        ),
        tags=[
            "AgentMart",
            "products",
            "recommendation",
            "pricing",
        ],
        examples=[
            (
                "Recommend noise-cancelling headphones below S$400. "
                "Battery life matters."
            )
        ],
    )

    return AgentCard(
        name="Product Advisor",
        description=(
            "AgentMart specialist for product discovery, comparison, "
            "recommendation, and pricing."
        ),
        url=public_url + "/",
        version="1.0.0",
        default_input_modes=["text"],
        default_output_modes=["text"],
        capabilities=AgentCapabilities(streaming=False),
        skills=[skill],
    )

In [20]:
def build_fulfilment_agent_card(public_url: str):
    skill = AgentSkill(
    id="fulfilment",
    name="Fulfilment",
    description=(
    "Checks AgentMart inventory and estimates Singapore delivery."
    ),
    tags=[
    "AgentMart",
    "inventory",
    "delivery",
    ],
    examples=[
    "Is AM-H100 in stock?",
    "Can AM-H100 be delivered to postal code 018989?",
    ],
    )
    return AgentCard(
    name="Fulfilment",
    description=(
    "AgentMart specialist for inventory and Singapore delivery."
    ),
    url=public_url + "/",
    version="1.0.0",
    default_input_modes=["text"],
    default_output_modes=["text"],
    capabilities=AgentCapabilities(streaming=False),
    skills=[skill],
    )


## 3.5 Build and start the two A2A services

In [19]:
def build_a2a_app(
    agent_card: AgentCard,
    agent_executor: AgentExecutor,
):
    request_handler = DefaultRequestHandler(
        agent_executor=agent_executor,
        task_store=InMemoryTaskStore(),
    )

    server = A2AStarletteApplication(
        agent_card=agent_card,
        http_handler=request_handler,
    )

    return server.build()


def wait_for_port(
    port: int,
    timeout_seconds: int = 15,
):
    deadline = time.time() + timeout_seconds

    while time.time() < deadline:
        try:
            with socket.create_connection(
                ("127.0.0.1", port),
                timeout=1,
            ):
                return
        except OSError:
            time.sleep(0.25)

    raise TimeoutError(
        f"Server on port {port} did not start."
    )


def start_a2a_server(
    name: str,
    app,
    port: int,
):
    config = uvicorn.Config(
        app,
        host="0.0.0.0",
        port=port,
        log_level="warning",
    )

    server = uvicorn.Server(config)

    thread = threading.Thread(
        target=server.run,
        daemon=True,
    )
    thread.start()

    wait_for_port(port)

    A2A_SERVERS[name] = server
    A2A_THREADS[name] = thread

    print(
        f"{name} local server: "
        f"http://127.0.0.1:{port}"
    )


def stop_checkpoint_3():
    for server in list(A2A_SERVERS.values()):
        server.should_exit = True

    for thread in list(A2A_THREADS.values()):
        thread.join(timeout=3)

    A2A_SERVERS.clear()
    A2A_THREADS.clear()

    for process in list(TUNNEL_PROCESSES.values()):
        if process.poll() is None:
            process.terminate()

            try:
                process.wait(timeout=3)
            except subprocess.TimeoutExpired:
                process.kill()

    TUNNEL_PROCESSES.clear()

    print("Checkpoint 3 servers and tunnels stopped.")

## 3.6 Start both public remote agents

Each team gets two temporary public URLs.

Keep this Colab runtime running while Checkpoints 4 and 5 use these agents.

In [26]:
if A2A_SERVERS or TUNNEL_PROCESSES:
    stop_checkpoint_3()


_, PRODUCT_PUBLIC_URL = start_quick_tunnel(
    name="product",
    port=PRODUCT_PORT,
)

_, FULFILMENT_PUBLIC_URL = start_quick_tunnel(
    name="fulfilment",
    port=FULFILMENT_PORT,
)


product_agent_card = build_product_agent_card(
    PRODUCT_PUBLIC_URL
)

fulfilment_agent_card = build_fulfilment_agent_card(
    FULFILMENT_PUBLIC_URL
)


product_a2a_app = build_a2a_app(
    product_agent_card,
    ProductAdvisorAgentExecutor(),
)

fulfilment_a2a_app = build_a2a_app(
    fulfilment_agent_card,
    FulfilmentAgentExecutor(),
)


start_a2a_server(
    name="product",
    app=product_a2a_app,
    port=PRODUCT_PORT,
)

start_a2a_server(
    name="fulfilment",
    app=fulfilment_a2a_app,
    port=FULFILMENT_PORT,
)


print("\n" + "=" * 70)
print("TEAM A2A ENDPOINTS — SAVE THESE FOR CHECKPOINTS 4, 5 AND DAY 3")
print("=" * 70)
print("Product Advisor:", PRODUCT_PUBLIC_URL)
print("Fulfilment     :", FULFILMENT_PUBLIC_URL)
print("=" * 70)

Checkpoint 3 servers and tunnels stopped.
product local server: http://127.0.0.1:9998
fulfilment local server: http://127.0.0.1:9999

TEAM A2A ENDPOINTS — SAVE THESE FOR CHECKPOINTS 4, 5 AND DAY 3
Product Advisor: https://idaho-compression-quickly-mit.trycloudflare.com
Fulfilment     : https://manager-enzyme-treated-annotated.trycloudflare.com


# Part C — Discover and Delegate to Remote Agents

## 3.7 A2A response parsing and discovery

**Format 1 — Direct Message**

{
  "result": {
    "role": "agent",
    "parts": [
      {
        "type": "text",
        "text": "AM-H100 is in stock."
      }
    ]
  }
}

**Format 2 — Task Status Message**

{
  "result": {
    "status": {
      "state": "completed",
      "message": {
        "parts": [
          {
            "type": "text",
            "text": "Task completed."
          }
        ]
      }
    }
  }
}

**Format 3 — Task Artifacts**
{
  "result": {
    "artifacts": [
      {
        "parts": [
          {
            "type": "text",
            "text": "Generated result"
          }
        ]
      }
    ]
  }
}

In [28]:
def _parts_to_text(parts: Any) -> str:
    if not isinstance(parts, list):
        return ""

    texts = []

    for part in parts:
        if not isinstance(part, dict):
            continue

        text = part.get("text")
        if isinstance(text, str) and text.strip():
            texts.append(text.strip())
            continue

        root = part.get("root")
        if isinstance(root, dict):
            root_text = root.get("text")
            if isinstance(root_text, str) and root_text.strip():
                texts.append(root_text.strip())

    return "\n".join(texts)


def _extract_response_text(data: Dict[str, Any]) -> str:
    result = data.get("result")

    if not isinstance(result, dict):
        return json.dumps(data, ensure_ascii=False)

    direct_text = _parts_to_text(result.get("parts"))
    if direct_text:
        return direct_text

    artifacts = result.get("artifacts")
    if isinstance(artifacts, list):
        texts = [
            _parts_to_text(artifact.get("parts"))
            for artifact in artifacts
            if isinstance(artifact, dict)
        ]
        artifact_text = "\n".join(
            text for text in texts if text
        )
        if artifact_text:
            return artifact_text

    status = result.get("status")
    if isinstance(status, dict):
        status_message = status.get("message")
        if isinstance(status_message, dict):
            status_text = _parts_to_text(
                status_message.get("parts")
            )
            if status_text:
                return status_text

    history = result.get("history")
    if isinstance(history, list):
        for message in reversed(history):
            if not isinstance(message, dict):
                continue

            role = str(
                message.get("role", "")
            ).lower()

            if role in {
                "agent",
                "assistant",
                "role_agent",
            }:
                history_text = _parts_to_text(
                    message.get("parts")
                )
                if history_text:
                    return history_text

    return json.dumps(result, ensure_ascii=False)


AGENT_ENDPOINTS = {
    "product_advisor": PRODUCT_PUBLIC_URL,
    "fulfilment": FULFILMENT_PUBLIC_URL,
}

AGENT_CARDS = {}


async def discover_agents():
    AGENT_CARDS.clear()

    async with httpx.AsyncClient(
        timeout=30.0
    ) as httpx_client:

        for agent_id, base_url in AGENT_ENDPOINTS.items():
            resolver = A2ACardResolver(
                httpx_client=httpx_client,
                base_url=base_url,
            )

            card = await resolver.get_agent_card()
            AGENT_CARDS[agent_id] = card

            print(
                f"Discovered {agent_id}: "
                f"{card.name}"
            )


await discover_agents()


AVAILABLE_AGENTS = "\n".join(
    (
        f"- {agent_id} ({card.name}): "
        f"{card.description}"
    )
    for agent_id, card in AGENT_CARDS.items()
)

print("\nAvailable remote agents:")
print(AVAILABLE_AGENTS)

Discovered product_advisor: Product Advisor
Discovered fulfilment: Fulfilment

Available remote agents:
- product_advisor (Product Advisor): AgentMart specialist for product discovery, comparison, recommendation, and pricing.
- fulfilment (Fulfilment): AgentMart specialist for inventory and Singapore delivery.


## 3.8 One generic A2A delegation tool

The Coordinator reasons about the **agents**.

`ask_remote_agent(...)` only performs the selected delegation.

In [29]:
from langchain_core.tools import tool


DELEGATION_TRACE = []


async def send_a2a_message(
    agent_id: str,
    text: str,
) -> str:
    if agent_id not in AGENT_CARDS:
        raise ValueError(
            f"Unknown agent_id={agent_id!r}. "
            f"Available: {sorted(AGENT_CARDS)}"
        )

    card = AGENT_CARDS[agent_id]

    async with httpx.AsyncClient(
        timeout=60.0
    ) as httpx_client:
        client = A2AClient(
            httpx_client=httpx_client,
            agent_card=card,
        )

        request = SendMessageRequest(
            id=str(uuid4()),
            params=MessageSendParams(
                message={
                    "role": "user",
                    "parts": [
                        {
                            "type": "text",
                            "text": text,
                        }
                    ],
                    "messageId": uuid4().hex,
                }
            ),
        )

        response = await client.send_message(
            request
        )

        data = response.model_dump(
            mode="json",
            exclude_none=True,
        )

    return _extract_response_text(data)


@tool
async def ask_remote_agent(
    agent_name: str,
    task: str,
) -> str:
    """Delegate a complete task to one discovered remote A2A agent.

    Args:
        agent_name: Stable ID of the selected remote agent.
        task: Complete task and all context needed by that remote agent.
    """
    if agent_name not in AGENT_CARDS:
        return (
            "DELEGATION_ERROR: unknown remote agent. "
            f"Available agents: {sorted(AGENT_CARDS)}"
        )

    print(
        f"\n[Coordinator → {agent_name}]"
    )
    print(task)

    response = await send_a2a_message(
        agent_name,
        task,
    )

    print(
        f"\n[{agent_name} → Coordinator]"
    )
    print(response)

    DELEGATION_TRACE.append(
        {
            "agent": agent_name,
            "task": task,
            "response": response,
        }
    )

    return response


print("Generic A2A delegation tool ready.")

Generic A2A delegation tool ready.


## 3.9 Build the Coordinator from discovered Agent Cards

In [30]:
COORDINATOR_PROMPT = f"""You are the AgentMart Coordinator Agent.

The following remote agents were discovered through A2A Agent Cards:

{AVAILABLE_AGENTS}

You have one generic delegation capability: ask_remote_agent(agent_name, task).

Your responsibility is to reason about which REMOTE AGENT should handle the
next task, then use the generic delegation capability to carry out that decision.

Rules:
- Choose an agent based on the discovered agent descriptions.
- Give the selected agent a complete, self-contained task.
- Do not invent AgentMart facts.
- For recommendation + availability, first obtain a recommendation and exact
  PRODUCT_ID, then delegate the availability check.
- For a direct stock/delivery question with a product ID, do not call Product
  Advisor unless product recommendation or product details are actually needed.
- If a recommended product is out of stock and the user requests alternatives,
  delegate again to Product Advisor with the rejected product IDs included,
  then check the new candidate.
- Never check the same rejected candidate twice.
- Stop after at most 3 candidate products when the user asks for alternatives.
- Return one concise customer-facing final answer.
- Do not expose internal delegation JSON in the final answer.
"""

coordinator_agent = create_agent(
    model=llm,
    tools=[ask_remote_agent],
    name="coordinator_agent",
    system_prompt=COORDINATOR_PROMPT,
)

print("A2A Coordinator ready.")

A2A Coordinator ready.


## 3.10 Run the normal AgentMart request

In [31]:
DELEGATION_TRACE.clear()

result = await coordinator_agent.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": DEFAULT_QUERY,
            }
        ]
    }
)

print("\n=== Delegation Trace ===")
for i, call in enumerate(
    DELEGATION_TRACE,
    start=1,
):
    print(
        f"{i}. {call['agent']}: "
        f"{call['task']}"
    )


[Coordinator → product_advisor]
Please recommend a pair of noise-cancelling headphones suitable for frequent travelers, with a budget of S$400 or less, prioritizing long battery life.


/tmp/ipykernel_1344/4102142583.py:22: DeprecationWarning: A2AClient is deprecated and will be removed in a future version. Use ClientFactory to create a client with a JSON-RPC transport.
  client = A2AClient(



=== Product Advisor A2A ===
query: Please recommend a pair of noise-cancelling headphones suitable for frequent travelers, with a budget of S$400 or less, prioritizing long battery life.
response: {"answer":"I recommend the **SonicVoyage Pro**. It features:\n- **Battery Life**: Up to 60 hours\n- **Noise Cancelling**: Adaptive active noise cancellation\n- **Weight**: 245 grams\n- **Additional Features**: Multipoint Bluetooth, folding travel case, quick charge\n- **Price**: S$379\n\nThis model is ideal for frequent travelers due to its long battery life and convenient travel features.","product_id":"AM-H100"}

[product_advisor → Coordinator]
{"answer":"I recommend the **SonicVoyage Pro**. It features:\n- **Battery Life**: Up to 60 hours\n- **Noise Cancelling**: Adaptive active noise cancellation\n- **Weight**: 245 grams\n- **Additional Features**: Multipoint Bluetooth, folding travel case, quick charge\n- **Price**: S$379\n\nThis model is ideal for frequent travelers due to its long bat

/tmp/ipykernel_1344/4102142583.py:22: DeprecationWarning: A2AClient is deprecated and will be removed in a future version. Use ClientFactory to create a client with a JSON-RPC transport.
  client = A2AClient(



=== Fulfilment A2A ===
query: Check the availability of the SonicVoyage Pro headphones (PRODUCT_ID: AM-H100) for delivery in Singapore.
response: {"answer":"The SonicVoyage Pro headphones are in stock.","in_stock":true}

[fulfilment → Coordinator]
{"answer":"The SonicVoyage Pro headphones are in stock.","in_stock":true}

=== Delegation Trace ===
1. product_advisor: Please recommend a pair of noise-cancelling headphones suitable for frequent travelers, with a budget of S$400 or less, prioritizing long battery life.
2. fulfilment: Check the availability of the SonicVoyage Pro headphones (PRODUCT_ID: AM-H100) for delivery in Singapore.


In [32]:
def pretty_print(response):
    for m in response["messages"]:
        print("\n---", m.type, "---")
        print(m)

    last_msg = response["messages"][-1]

    if isinstance(last_msg.content, list):
        text = "".join(
            block["text"]
            for block in last_msg.content
            if block.get("type") == "text"
        )
    else:
        text = last_msg.content
    print(text)
pretty_print(result)


--- human ---
content='I need a pair of noise-cancelling headphones below S$400. I travel frequently, so battery life matters. Recommend one and tell me whether it is available.' additional_kwargs={} response_metadata={} id='35b0a9a5-950f-4e20-917c-bed8bb35a967'

--- ai ---
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 388, 'total_tokens': 440, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 8.94e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 8.94e-05, 'upstream_inference_prompt_cost': 5.82e-05, 'upstream_inference_completions_cost': 3.12e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_cd839501f6', 'id': 'gen-1